In [ ]:
# Re-import necessary libraries since execution state was reset
import pandas as pd
from sklearn.metrics import f1_score, cohen_kappa_score

def calculate_metrics(df, col1, col2):
    """
    Calculate mismatch percentage, F1-score, and Cohen's Kappa for two categorical columns.
    
    Parameters:
    df (pd.DataFrame): DataFrame containing the columns to compare.
    col1 (str): Name of the first column (ground truth labels).
    col2 (str): Name of the second column (predicted labels).

    Returns:
    dict: A dictionary containing mismatch percentage, F1-score, and Cohen's Kappa score.
    """
    # Define valid labels
    valid_values = ['Sexual Harresment', 'sexual harresment', 'Violense', 'violense', 
                    'Harmful', 'harmful', 'Offensive', 'offensive', 'Propaganda', 'propaganda', 
                    'Not Hateful', 'not hateful']

    # df[col1] = df[col1].apply(safe_first_label)
    # df[col2] = df[col2].apply(safe_first_label)

    df[col1] = df[col1].str.strip("'")
    # df[col2] = df[col2].str.strip("'")
    df[col2] = df[col2].astype(str).str.strip().str.strip("'").str.lower()
    # print(df[col1].head(5))
    # print(df[col2].head(5))
    # Filter valid rows
    filtered_df = df[(df[col1].str.lower().isin([v.lower() for v in valid_values])) & 
                     (df[col2].str.lower().isin([v.lower() for v in valid_values]))]

    # Exclude rows where col2 has 'Error' or 'None'
    filtered_df = filtered_df[~filtered_df[col2].isin(['Error', 'None'])]

    # Total valid rows
    total_valid_rows = len(filtered_df)
    if total_valid_rows == 0:
        return {"Mismatch Percentage": 0.0, "F1 Score": 0.0, "Cohen's Kappa": 0.0}

    # Calculate mismatch percentage
    matches = (filtered_df[col1].str.lower() == filtered_df[col2].str.lower()).sum()
    #mismatch_percentage = (matches / total_valid_rows) * 100
    mismatch_percentage = ((total_valid_rows-matches) / total_valid_rows) * 100
    # Convert labels to categorical integers for sklearn metrics
    unique_labels = sorted(set(filtered_df[col1].str.lower()).union(set(filtered_df[col2].str.lower())))
    label_to_int = {label: idx for idx, label in enumerate(unique_labels)}

    y_true = filtered_df[col1].str.lower().map(label_to_int).values
    y_pred = filtered_df[col2].str.lower().map(label_to_int).values

    # Calculate F1-score (macro for balanced importance across all labels)
    f1 = f1_score(y_true, y_pred, average='macro')

    # Calculate Cohen's Kappa
    kappa = cohen_kappa_score(y_true, y_pred)
    print(f"Mismatch Percentage: {mismatch_percentage}")
    print(f"F1 Score: {f1}")
    print(f"Cohen's Kappa: {kappa}")
    return {
        "Mismatch Percentage": mismatch_percentage,
        "F1 Score": f1,
        "Cohen's Kappa": kappa
    }

# This function is now ready to be used with a dataset. Let me know if you want me to apply it to a specific file.


In [ ]:
# import pandas as pd
# from bert_score import score as bert_score
# from sentence_transformers import SentenceTransformer
# from sklearn.metrics.pairwise import cosine_similarity
# import numpy as np
# #model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:


# def calculate_metrics(df, true_col, pred_col):
#     # Extract text lists
#     true_texts = df[true_col].astype(str).tolist()
#     pred_texts = df[pred_col].astype(str).tolist()

#     # BERTScore (returns precision, recall, f1)
#     P, R, F1 = bert_score(
#         pred_texts, true_texts,
#         lang="bn",  # Bengali
#         model_type="xlm-roberta-base",  # You can also try 'csebuetnlp/banglabert'
#         verbose=False,
#         #progress_bar=False
#     )
#     bert_scores = F1.tolist()

#     # Cosine Similarity using Sentence Transformers
    
#     embedder = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")
#     emb_true = embedder.encode(true_texts, convert_to_tensor=True)
#     emb_pred = embedder.encode(pred_texts, convert_to_tensor=True)
#     cosine_scores = np.diag(cosine_similarity(emb_true.cpu().numpy(), emb_pred.cpu().numpy())).tolist()

#     # Combine results
#     result_df = df.copy()
#     result_df["BERTScore_F1"] = bert_scores
#     result_df["Cosine_Similarity"] = cosine_scores

#     # Print average metrics
#     print(f"Average BERTScore (F1): {np.mean(bert_scores):.4f}")
#     print(f"Average Cosine Similarity: {np.mean(cosine_scores):.4f}")

#     return result_df


In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score
from difflib import get_close_matches

def extract_closest_label(text, valid_values, cutoff=0.6):
    """
    Extract the closest valid label from a text using fuzzy matching.
    """
    matches = get_close_matches(text.lower(), [v.lower() for v in valid_values], n=1, cutoff=cutoff)
    return matches[0] if matches else None

def calculate_metrics2(df1, df2, col1, col2, sample_size=1000, random_state=42):
    """
    Calculate mismatch percentage, F1-score, and Cohen's Kappa for two categorical columns
    from two different dataframes using a random sample.
    """
    # Valid labels
    valid_values = [
        'Sexual Harresment', 'sexual harresment',
        'Violense', 'violense',
        'Harmful', 'harmful',
        'Propaganda', 'propaganda',
        'Not Hateful', 'not hateful'
    ]

    # Lowercase all labels and remove extra quotes from df1 and df2
    df1_clean = df1.copy()
    df2_clean = df2.copy()

    df1_clean[col1] = df1_clean[col1].fillna('').str.strip("'").str.lower()
    df2_clean[col2] = df2_clean[col2].fillna('').str.strip().str.lower()

    # Apply fuzzy matching to df2[col2]
    df2_clean[col2] = df2_clean[col2].apply(lambda x: extract_closest_label(x, valid_values))

    # Filter valid rows: both true and pred must be in valid labels
    valid_labels_lower = [v.lower() for v in valid_values]
    filtered_indices = df1_clean[
        (df1_clean[col1].isin(valid_labels_lower)) & 
        (df2_clean[col2].isin(valid_labels_lower))
    ].index

    # Subset the data
    if len(filtered_indices) < sample_size:
        sample_size = len(filtered_indices)  # adjust if not enough samples

    np.random.seed(random_state)
    sampled_indices = np.random.choice(filtered_indices, size=sample_size, replace=False)

    y_true = df1_clean.loc[sampled_indices, col1].values
    y_pred = df2_clean.loc[sampled_indices, col2].values

    # Calculate mismatch percentage
    matches = (y_true == y_pred).sum()
    mismatch_percentage = ((sample_size - matches) / sample_size) * 100

    # Encode labels to integers
    unique_labels = sorted(set(list(y_true) + list(y_pred)))
    label_to_int = {label: idx for idx, label in enumerate(unique_labels)}

    y_true_int = [label_to_int[label] for label in y_true]
    y_pred_int = [label_to_int[label] for label in y_pred]

    # Calculate metrics
    f1 = f1_score(y_true_int, y_pred_int, average='macro')
    kappa = cohen_kappa_score(y_true_int, y_pred_int)

    return {
        "Mismatch Percentage": mismatch_percentage,
        "F1 Score": f1,
        "Cohen's Kappa": kappa,
        "Sample Size": sample_size
    }



In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

In [ ]:
# # Load the CSV file
csv_file_path = '/kaggle/input/reduced/Reducing labels/Gold Labels/BanglaHateSpeech_GPT-4.1 (5 labels).csv'  # Replace with your CSV file path
# # with open(csv_file_path, "rb") as f:
# #     result = chardet.detect(f.read())
#     #print(result)
df1 = pd.read_csv(csv_file_path, encoding='ISO-8859-1')
# #, encoding='ISO-8859-1'

In [ ]:
for column in df.columns:
    print(column)

In [ ]:
# #Reasoning evaluation
evaluation = calculate_metrics2(df1, df, 'Plitical Personality_biased Label', 'Plitical Personality_biased Label')

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score, cohen_kappa_score

# Load files
file1 = "/kaggle/input/reduced/Reducing labels/Qwen 2.5 7B/BanglaHateSpeech_Qwen 2.5 7B (2 labels).csv"
file2 = "/kaggle/input/reduced/Reducing labels/Gold Labels/BanglaHateSpeech_GPT-4.1 (2 labels).csv"

df1 = pd.read_csv(file1)
df2 = pd.read_csv(file2)

# Preprocess: remove brackets/quotes
df1['Plitical Personality_biased Label'] = df1['NonPlitical Personality_biased Label'].astype(str).str.strip("[]'\" ")
df2['NonPlitical Personality_biased Label'] = df2['NonPlitical Personality_biased Label'].astype(str).str.strip("[]'\" ")

# Drop rows where label is missing or corrupted
valid_mask = (
    df1['Plitical Personality_biased Label'].notna() &
    df2['NonPlitical Personality_biased Label'].notna() &
    (df2['Plitical Personality_biased Label'] != "Understood. Please provide the [text] to") &
    (df2['NonPlitical Personality_biased Label'] != "Understood. Please provide the [text] to") &
    (df1['Plitical Personality_biased Label'] != "") &
    (df2['NonPlitical Personality_biased Label'] != "")
)

df1_clean = df1[valid_mask].reset_index(drop=True)
df2_clean = df2[valid_mask].reset_index(drop=True)

# Final comparison
y_true = df1_clean['Plitical Personality_biased Label']
y_pred = df2_clean['NonPlitical Personality_biased Label']

f1 = f1_score(y_true, y_pred, average='weighted')
kappa = cohen_kappa_score(y_true, y_pred)
mismatches = (y_true != y_pred).sum()
total_count = len(y_true)
mismatch_percentage = (mismatches / total_count) * 100
print("F1 Score:", f1)
print("Cohen's Kappa:", kappa)
print("Mismatch Count:", mismatch_percentage)


In [ ]:
print(evaluation)

In [ ]:
evaluation = calculate_metrics(df, 'Final Label', 'Not Female_biased')

In [ ]:
evaluation = calculate_metrics(df, 'Female_biased', 'Not Female_biased')

In [ ]:
# List of all target columns (excluding Reasoning & input columns like Sample_text, Category, etc.)
target_columns = [
    #"Female_biased", "Not Female_biased",
    # "Artist_biased Label", "Not Artist_biased Label",
    # "Sportsman_biased Label", "Not Sports_biased Label",
    # "Vlogger_biased Label", "Not Vlogger_biased Label",
    # "Muslim_biased Label", "Hindu_biased Label", "Not Muslim and Hindu",
    # "Diplomacy_biased Label", "Not Diplomacy_biased Label",
    # "Security_Analyst_biased Label", "Not Security_Analyst_biased Label",
    # "Journalist_biased Label", "Not Journalist_biased Label",
    "Plitical Personality_biased Label", "NonPlitical Personality_biased Label",
    # "International Relation Analyst_biased Label", "Not International Relation Analyst_biased Label"
]

# Dictionary to store evaluation results
all_evaluations = {}

# Loop through all target columns
for col in target_columns:
    print(f"\nEvaluating: {col}")
    try:
        result = calculate_metrics(df, 'Final Label', col)
        
        all_evaluations[col] = result
    except Exception as e:
        print(f"Error while evaluating {col}: {e}")
